In [1]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import zipfile
import os
import json
from datasets import load_dataset,Dataset, DatasetDict
import torch
import numpy as np
import evaluate

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"maiisediik","key":"b0d5feda5c32ddf3454d7a3ab78bc4bc"}'}

In [4]:
! kaggle datasets download -d "stanfordu/stanford-question-answering-dataset"

Dataset URL: https://www.kaggle.com/datasets/stanfordu/stanford-question-answering-dataset
License(s): CC-BY-SA-4.0
100% 8.73M/8.73M [00:00<00:00, 63.4MB/s]



In [5]:
with zipfile.ZipFile("/content/stanford-question-answering-dataset.zip","r") as zipref:
  zipref.extractall()

In [6]:
def load_json(path):
    with open(path) as f:
        squad = json.load(f)
        return squad

In [7]:
s=load_json("dev-v1.1.json")

In [8]:
s.keys()

dict_keys(['data', 'version'])

In [9]:
s['data'][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
s['data'][0]['paragraphs'][0].keys()

dict_keys(['context', 'qas'])

In [11]:
s['data'][0]['paragraphs'][0]['qas'][0].keys()

dict_keys(['answers', 'question', 'id'])

In [12]:
s['data'][0]['paragraphs'][0]['qas'][0]['answers'][0].keys()

dict_keys(['answer_start', 'text'])

In [13]:
s['data'][0]['paragraphs'][0]

{'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
 'qas': [{'answers': [{'answer_start': 177, 'text': 'Denver Broncos'},
    {'answer_start': 177, 'text': 'Denver Broncos'},
    {'answer_start': 177, 'text': 'Denver Broncos'}],
   'question': 'Which NFL team

In [14]:
def load_json_train(path):
    with open(path) as f:
        squad = json.load(f)
    examples = []
    for article in squad["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                    examples.append({
                        "id": qa["id"],
                        "question": qa["question"],
                        "context": context,
                        "answers": [a["text"] for a in qa["answers"]][0],
                        "answer_starts": [a["answer_start"] for a in qa["answers"]][0]
                   })
    return examples

In [15]:
def load_json_val(path):
    with open(path) as f:
        squad = json.load(f)
    examples = []
    for article in squad["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                    examples.append({
                        "id": qa["id"],
                        "question": qa["question"],
                        "context": context,
                        "answers": [a["text"] for a in qa["answers"]],
                        "answer_starts": [a["answer_start"] for a in qa["answers"]],
                    })
    return examples

In [16]:
train_examples = load_json_train("train-v1.1.json")


In [17]:
dev_examples = load_json_val("dev-v1.1.json")

In [18]:
train_examples[0]['question']

'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'

In [19]:
train_examples[0]['context']

'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [20]:
train_examples[0]['answers']

'Saint Bernadette Soubirous'

In [21]:
train_examples = pd.DataFrame(train_examples)
dev_examples = pd.DataFrame(dev_examples)

In [22]:
train_examples

,id,question,context,answers,answer_starts
0,5733be284776f41900661182,To whom did the Virgin Mary allegedly appear i...,"Architecturally, the school has a Catholic cha...",Saint Bernadette Soubirous,515
1,5733be284776f4190066117f,What is in front of the Notre Dame Main Building?,"Architecturally, the school has a Catholic cha...",a copper statue of Christ,188
2,5733be284776f41900661180,The Basilica of the Sacred heart at Notre Dame...,"Architecturally, the school has a Catholic cha...",the Main Building,279
3,5733be284776f41900661181,What is the Grotto at Notre Dame?,"Architecturally, the school has a Catholic cha...",a Marian place of prayer and reflection,381
4,5733be284776f4190066117e,What sits on top of the Main Building at Notre...,"Architecturally, the school has a Catholic cha...",a golden statue of the Virgin Mary,92
...,...,...,...,...,...
87594,5735d259012e2f140011a09d,In what US state did Kathmandu first establish...,"Kathmandu Metropolitan City (KMC), in order to...",Oregon,229
87595,5735d259012e2f140011a09e,What was Yangon previously known as?,"Kathmandu Metropolitan City (KMC), in order to...",Rangoon,414
87596,5735d259012e2f140011a09f,With what Belorussian city does Kathmandu have...,"Kathmandu Metropolitan City (KMC), in order to...",Minsk,476
87597,5735d259012e2f140011a0a0,In what year did Kathmandu create its initial ...,"Kathmandu Metropolitan City (KMC), in order to...",1975,199


In [23]:
# already have the data loaded in Python as pandas DataFrames
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_examples, preserve_index=False),
    "val": Dataset.from_pandas(dev_examples, preserve_index=False),
})

In [24]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [25]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModelForQuestionAnswering.from_pretrained("roberta-base")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForQuestionAnswering LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
qa_outputs.bias           | MISSING    | 
qa_outputs.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
max_length = 512
doc_stride = 128

In [27]:
tokenizer.model_max_length

512

In [28]:
def prepare_train_features(examples):
    questions = [q.strip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",       # only truncate the context, never the question
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True, # long contexts -> multiple windows
        return_offsets_mapping=True,    # char offsets per token, needed below
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []


    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]  #[0,0,0,1,1]
        answer_text = examples["answers"][sample_idx]
        start_char = examples["answer_starts"][sample_idx]
        end_char = start_char + len(answer_text)

        sequence_ids = tokenized.sequence_ids(i)

        # find the token span that belongs to the context (sequence_id == 1)
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # if the answer isn't fully inside this window, label it as unanswerable (CLS token, index 0)
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

In [29]:
train_tokenized = dataset['train'].map(
    prepare_train_features, batched=True,
    remove_columns=dataset['train'].column_names
)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

In [30]:
def prepare_validation_features(data):
    questions = [q.strip() for q in data["question"]]
    tokenized = tokenizer(
        questions,
        data["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True, #If the question + context is too long for max_length, do not just cut the extra context and lose it. Instead, create extra chunks/windows for the overflow part.
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping") #chunks of long context
    example_ids = []
    for i in range(len(tokenized["input_ids"])): #input_ids[0] = question + context window 1
        sample_idx = sample_map[i]    #[0,0,0,1,1]
        example_ids.append(data["id"][sample_idx])
        sequence_ids = tokenized.sequence_ids(i) #[none,0,0,0,0,0,none,1,1,1,1,1,1,none]
        # keep offsets only for context tokens, null out the rest
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]
    tokenized["example_id"] = example_ids #So now each input_ids[i] has an example_id[i] come from which question
    return tokenized

In [31]:
val_tokenized = dataset['val'].map(
    prepare_validation_features, batched=True,
    remove_columns=dataset['val'].column_names
)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [32]:
from transformers import TrainingArguments, Trainer

In [33]:
arg = TrainingArguments(
    output_dir='results',
    save_strategy="epoch",
    save_total_limit=2,
    weight_decay=0.01,
    learning_rate=0.00002,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          # SQuAD fine-tuning standard is 2-3 epochs;
    fp16=True,                   # L4 has strong fp16 tensor-core throughput
    remove_unused_columns=False, # keep offset_mapping / example_id alive for manual postprocessing below
)

In [34]:
metric = evaluate.load("squad")

In [35]:
def postprocess_qa_predictions(examples, features, raw_predictions, n_best=20, max_answer_length=30): #20 * 20 = 400 answer spans are tested per window.
    all_start_logits, all_end_logits = raw_predictions #all_start_logits, all_end_logits
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}      #example_id_to_index = {"q1": 0 ,"q2": 1}

    features_per_example = {} #for each original example, which tokenized windows belong to it?

    for i, feat in enumerate(features):
        features_per_example.setdefault(example_id_to_index[feat["example_id"]], []).append(i) #This groups windows by original example.
        # features_per_example = {0: [0, 1],  # q1 has windows 0 and 1  1: [2]   , # q2 has window 2}

    predictions = {}

    for ex_idx, example in enumerate(examples):
        context = example["context"]
        best_score, best_text = -1e9, "" #Start with a very bad score and empty answer.

        for feat_idx in features_per_example[ex_idx]: # Get model scores for this window.
            start_logits = all_start_logits[feat_idx] #start_logits[token] = how likely this token is the start.
            end_logits = all_end_logits[feat_idx]     #end_logits[token] = how likely this token is the end.

            offsets = features[feat_idx]["offset_mapping"]  # Get offset mapping for this window. note features is list of dictionary

            start_indexes = np.argsort(start_logits)[-n_best:][::-1] #This gets the top n_best start token indexes. reverses them:[::-1] . [-n_best] mean last 20
            end_indexes = np.argsort(end_logits)[-n_best:][::-1]     #Same thing, but for end tokens.

            #Try every combination of possible start and end.
            for s in start_indexes:
                for e in end_indexes:
                    if offsets[s] is None or offsets[e] is None: # imagine the model gives high score to token 3, which is "Ahmed" in the question, not the context.
                                                                  # That would be bad, because answer should be extracted from the context
                        continue
                    if e < s or e - s + 1 > max_answer_length: #means end token comes before start token. Invalid. or answer is too long. For SQuAD, 30 is commonly used because most answers are shorter than 30 tokens.
                        continue #skip

                    score = start_logits[s] + end_logits[e]  # If the model is confident that token s is the start and token e is the end, this score will be high.
                    if score > best_score:  # If this answer span is better than all previous spans, save it.
                        best_score = score
                        best_text = context[offsets[s][0]:offsets[e][1]]  #Convert token span back to text
        predictions[example["id"]] = best_text

    return predictions # { "q1": "New York City", "q2": "OpenAI" }


In [36]:
trainer = Trainer(
    model=model,
    args=arg,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,

)

In [37]:
trainer.train()

Step,Training Loss
500,1.822193
1000,1.193605
1500,1.103611
2000,1.025736
2500,0.983082
3000,0.972889
3500,0.954434
4000,0.920594
4500,0.921621
5000,0.907837


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16452, training_loss=0.7711783737676033, metrics={'train_runtime': 1287.0989, 'train_samples_per_second': 204.506, 'train_steps_per_second': 12.782, 'total_flos': 6.877853230804992e+16, 'train_loss': 0.7711783737676033, 'epoch': 3.0})

In [46]:
# 1. Prepare dataset for prediction by removing non-tensor features
eval_dataset = val_tokenized.remove_columns(
    [col for col in val_tokenized.column_names if col not in ['input_ids', 'attention_mask', 'token_type_ids']]
)

# 2. Predict raw start and end logits
raw_preds = trainer.predict(eval_dataset)

# 3. Convert logits back to character answers
preds = postprocess_qa_predictions(
    examples=dataset['val'],
    features=val_tokenized,
    raw_predictions=raw_preds.predictions
)

# 4. Format predictions
formatted_preds = [{"id": k, "prediction_text": v} for k, v in preds.items()]

# 5. Format references to match the required dictionary structure
references = [
    {
        "id": ex["id"],
        "answers": {
            "text": ex["answers"] if isinstance(ex["answers"], list) else ex["answers"]["text"],
            "answer_start": ex.get("answer_starts", [0] * len(ex["answers"])) if isinstance(ex["answers"], list) else ex["answers"]["answer_start"]
        }
    }
    for ex in dataset['val']
]

# 6. Compute metrics
result = metric.compute(predictions=formatted_preds, references=references)
print(result)

{'exact_match': 85.9035004730369, 'f1': 92.12806322785674}


In [39]:
# Save the fine-tuned model + tokenizer to a clean folder, ready to push to GitHub for inference
save_dir = "qa-roberta-squad-best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Saved best QA checkpoint to ./{save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best QA checkpoint to ./qa-roberta-squad-best


##Inferrence

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def answer_question(question, context, n_best=20, max_answer_length=30):
    inputs = tokenizer(
        question.strip(),
        context,
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,   # same as validation: handles long contexts via multiple windows
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")  # not used further, just consistent w/ validation prep

    inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs_on_device)

    all_start_logits = outputs.start_logits.cpu().numpy()
    all_end_logits = outputs.end_logits.cpu().numpy()

    best_score, best_text = -1e9, ""

    for i in range(all_start_logits.shape[0]):  # loop over windows (usually 1, more if context is long)
        sequence_ids = inputs.sequence_ids(i)
        offsets = [
            o if sequence_ids[k] == 1 else None   # null out non-context tokens, same as prepare_validation_features
            for k, o in enumerate(offset_mapping[i].tolist())
        ]

        start_logits = all_start_logits[i]
        end_logits = all_end_logits[i]

        start_indexes = np.argsort(start_logits)[-n_best:][::-1]
        end_indexes = np.argsort(end_logits)[-n_best:][::-1]

        for s in start_indexes:
            for e in end_indexes:
                if offsets[s] is None or offsets[e] is None:
                    continue
                if e < s or e - s + 1 > max_answer_length:
                    continue
                score = start_logits[s] + end_logits[e]
                if score > best_score:
                    best_score = score
                    best_text = context[offsets[s][0]:offsets[e][1]]

    return best_text


In [41]:
question = "What dataset was used to train the model?"
context = "The model was fine-tuned on SQuAD v1.1, a reading comprehension dataset consisting of questions posed by crowdworkers on Wikipedia articles."

answer = answer_question(question, context)
print("Question:", question)
print("Answer:", answer)

Question: What dataset was used to train the model?
Answer: SQuAD v1.1


In [47]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a destination folder in your Drive
drive_path = '/content/drive/MyDrive/qa_best_model'
os.makedirs(drive_path, exist_ok=True)

Mounted at /content/drive


In [48]:
import shutil
import torch

# A. Save the PyTorch state dict (.pt) directly
pt_file_path = os.path.join(drive_path, "model.pt")
torch.save(trainer.model.state_dict(), pt_file_path)

# B. Save the complete Hugging Face model & tokenizer structure to the same Drive folder
trainer.save_model(drive_path)
tokenizer.save_pretrained(drive_path)

print(f"Successfully saved .pt file and best model artifacts to: {drive_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully saved .pt file and best model artifacts to: /content/drive/MyDrive/qa_best_model
